# Plotly + Streamlit + Storytelling with Data
** Data Visualization (Part 2)**

Three things in one notebook:
1. **Plotly** — interactive charts for EDA and dashboards
2. **Streamlit** — quick reference, not deep study
3. **Storytelling with data** — principles you internalize, not memorize

---
## PART 1: Plotly

### Why Plotly over Matplotlib for interactive work?
- Matplotlib = static images (great for papers, reports, model logs)
- Plotly = interactive HTML charts (hover, zoom, filter) — great for EDA, dashboards, presenting to stakeholders

### Two APIs to know:
- `plotly.express` (px) — high level, one-liners, use this 90% of the time
- `plotly.graph_objects` (go) — low level, full control, use when px can't do it

In [2]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# Sample data we'll reuse throughout
np.random.seed(42)
df = pd.DataFrame({
    'age':        np.random.randint(22, 60, 200),
    'salary':     np.random.normal(70000, 15000, 200),
    'experience': np.random.randint(1, 20, 200),
    'department': np.random.choice(['Engineering', 'Sales', 'Marketing', 'HR'], 200),
    'performance':np.random.choice(['Low', 'Mid', 'High'], 200),
    'churn':      np.random.choice([0, 1], 200, p=[0.75, 0.25])
})

### 1.1 The Charts You'll Actually Use in ML Work

In [ ]:
# --- Scatter plot (feature relationships, embeddings) ---
fig = px.scatter(
    df,                          # the dataframe — one row = one dot
    x='experience',              # dot's horizontal position
    y='salary',                  # dot's vertical position
    color='department',          # dot's color (4 departments = 4 colors)
    size='age',                  # dot's size (older person = bigger dot)
    hover_data=['performance'],  # extra info shown when you hover over a dot
    title='Salary vs Experience by Department'
)
fig.show()

In [5]:
# --- Histogram (distribution check — do before any ML) ---
fig = px.histogram(
    df,                    # the dataframe
    x='salary',            # which column to distribute into bins
    color='department',    # draw a separate histogram for each department
    barmode='overlay',     # stack bars on top of each other (vs 'group' = side by side)
    nbins=30,              # divide salary range into 30 bins
    opacity=0.7,           # make bars slightly transparent so overlaps are visible
    title='Salary Distribution by Department'
)

fig.show()

In [7]:
# --- Box plot (outlier detection, group comparison) ---
fig = px.box(
    df,
    x='department',      # groups on x-axis (one set of boxes per department)
    y='salary',          # what we're measuring
    color='performance', # split each department further by performance level
    title='Salary Distribution: Department X Performance'
)
fig.show()

In [8]:
# --- Bar chart (category comparisons, feature importance) ---
# group by department, compute mean salary, sort highest to lowest
dept_avg = df.groupby('department')['salary'].mean().reset_index().sort_values('salary', ascending=False)

fig = px.bar(
    dept_avg,
    x='department',   # departments on x-axis
    y='salary',       # bar height = average salary
    text='salary',    # raw value to display on each bar (formatted below)
    title='Average Salary by Department'
)

fig.update_traces(
    texttemplate='$%{text:,.0f}',  # format: $ sign, comma separator, 0 decimals
    textposition='outside'         # place the label above the bar, not inside
)

fig.show()

In [9]:
# --- Heatmap (correlation matrix — critical for feature selection) ---
# compute correlation matrix — every feature vs every other feature
# result is a 3×3 DataFrame where each cell = correlation coefficient
corr = df[['age', 'salary', 'experience']].corr()

fig = px.imshow(
    corr,
    text_auto='.2f',              # automatically show values on each cell, 2 decimal places
    color_continuous_scale='RdBu_r',  # red = positive correlation, blue = negative
    zmin=-1, zmax=1,              # fix color scale to full range [-1, 1]
                                  # without this, scale adjusts to your data range
                                  # e.g. if max corr is 0.3, red would mean 0.3 not 1.0
    title='Correlation Matrix'
)
fig.show()


In [10]:
# --- Line chart (training curves, time series, model metrics over epochs) ---
epochs = list(range(1, 21))
train_loss = [1.0 * (0.85 ** i) + np.random.normal(0, 0.02) for i in epochs]
val_loss   = [1.0 * (0.87 ** i) + np.random.normal(0, 0.03) for i in epochs]

fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=train_loss, mode='lines+markers', name='Train Loss'))
fig.add_trace(go.Scatter(x=epochs, y=val_loss,   mode='lines+markers', name='Val Loss',
                         line=dict(dash='dash')))
fig.update_layout(title='Training vs Validation Loss', xaxis_title='Epoch', yaxis_title='Loss')
fig.show()

## `px` vs `go`

- **`px` (plotly.express)** — builds a complete figure in one call. Use for single charts. Less code.
- **`go` (plotly.graph_objects)** — builds individual chart objects you manually place into a grid.

**Rule:**
- Single chart → `px`
- Multiple charts in one figure → `make_subplots` + `go`

```python
# px — one call, done
fig = px.histogram(df, x='salary')

# go — build piece by piece, place into grid
fig = make_subplots(rows=1, cols=2)
fig.add_trace(go.Histogram(...), row=1, col=1)
fig.add_trace(go.Scatter(...),   row=1, col=2)


### 1.2 Subplots — Multiple Charts in One View

In [12]:
# Useful for model evaluation dashboards
# make_subplots creates an empty grid to place charts into
# rows=1, cols=2 → one row, two columns (side by side)
# subplot_titles → label above each panel
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Salary Distribution', 'Experience vs Salary']
)

# add_trace places a chart into a specific cell of the grid
# row=1, col=1 → goes in the left panel
fig.add_trace(
    go.Histogram(x=df['salary'], nbinsx=20, name='Salary'),
    row=1, col=1
)

# row=1, col=2 → goes in the right panel
fig.add_trace(
    go.Scatter(x=df['experience'], y=df['salary'], mode='markers', name='Scatter'),
    row=1, col=2
)

fig.update_layout(title='Side-by-Side View', height=400)
fig.show()


### 1.3 Layout Customization (the bits you'll actually need)

In [13]:
fig = px.scatter(df, x='experience', y='salary', color='department')

fig.update_layout(
    title=dict(text='My Chart Title', font=dict(size=18)),  # title with custom font size
    xaxis_title='Years of Experience',  # x-axis label
    yaxis_title='Annual Salary ($)',    # y-axis label
    legend_title='Department',          # legend header text
    width=800, height=500,              # figure size in pixels
    template='plotly_white',            # clean white background — use over default gray
)

# add_hline draws a horizontal line across the entire chart at a given y value
# line_dash='dash' → dashed line style
# annotation_text  → label shown next to the line
fig.add_hline(
    y=df['salary'].mean(),
    line_dash='dash',
    line_color='red',
    annotation_text='Average'
)

fig.show()


### 1.4 Saving Charts

In [ ]:
# Save as interactive HTML (shareable, no server needed)
# fig.write_html('chart.html')

# Save as static image (needs kaleido: pip install kaleido)
# fig.write_image('chart.png', width=1200, height=600, scale=2)

---
## PART 2: Streamlit — Quick Reference

> **How to learn Streamlit:** Build something with it, not by studying it.
> This section is a reference card — come back to it when you're building a project.

```bash
pip install streamlit
streamlit run app.py
```

### The ~10 things that cover 80% of Streamlit usage:

```python
import streamlit as st
import pandas as pd
import plotly.express as px

# --- TEXT ---
st.title('My ML Dashboard')
st.header('Model Results')
st.subheader('Confusion Matrix')
st.write('Supports **markdown** and dataframes directly')
st.markdown('## Any markdown here')
st.metric(label='Accuracy', value='94.3%', delta='+2.1%')  # nice KPI card

# --- DATA DISPLAY ---
st.dataframe(df)              # interactive scrollable table
st.table(df.head())           # static table
st.json({'key': 'value'})     # for API responses, configs

# --- CHARTS ---
st.pyplot(matplotlib_fig)     # matplotlib/seaborn figures
st.plotly_chart(plotly_fig)   # plotly figures

# --- WIDGETS (user input) ---
threshold = st.slider('Decision Threshold', 0.0, 1.0, 0.5)
model     = st.selectbox('Select Model', ['Random Forest', 'XGBoost', 'Logistic Regression'])
features  = st.multiselect('Features', df.columns.tolist())
run_btn   = st.button('Run Model')
if run_btn:
    st.write('Running...')

# --- LAYOUT ---
col1, col2 = st.columns(2)    # side-by-side layout
with col1:
    st.metric('Precision', '0.92')
with col2:
    st.metric('Recall', '0.88')

with st.sidebar:
    st.header('Controls')
    n_estimators = st.slider('Trees', 10, 500, 100)

with st.expander('Show raw data'):   # collapsible section
    st.dataframe(df)

# --- PERFORMANCE ---
@st.cache_data                # cache expensive operations
def load_data(path):
    return pd.read_csv(path)

@st.cache_resource            # cache ML models (loaded once)
def load_model():
    import joblib
    return joblib.load('model.pkl')

# --- FEEDBACK ---
st.success('Model trained successfully!')
st.warning('Low sample size — interpret with caution')
st.error('Feature X not found in dataset')
st.info('Using default hyperparameters')
with st.spinner('Training...'):
    pass  # long operation here
```

### The mental model for Streamlit:
Streamlit re-runs your entire script top to bottom every time a widget changes.
That's it. Everything else follows from this. Use `@st.cache_data` to avoid re-running expensive code on every interaction.

### Minimal working ML app skeleton:
```python
import streamlit as st
import pandas as pd
import plotly.express as px
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

st.title('ML Model Explorer')

@st.cache_data
def load_data():
    return pd.read_csv('data.csv')

df = load_data()

with st.sidebar:
    n_estimators = st.slider('Number of Trees', 10, 300, 100)
    target = st.selectbox('Target Column', df.columns)

if st.button('Train Model'):
    X = df.drop(columns=[target])
    y = df[target]
    model = RandomForestClassifier(n_estimators=n_estimators)
    model.fit(X, y)
    acc = accuracy_score(y, model.predict(X))
    st.metric('Train Accuracy', f'{acc:.2%}')
    
    importance_df = pd.DataFrame({'feature': X.columns, 'importance': model.feature_importances_})
    fig = px.bar(importance_df.sort_values('importance'), x='importance', y='feature', orientation='h')
    st.plotly_chart(fig)
```

---
## PART 3: Storytelling with Data

> The goal is not to show all the data — it's to make one point undeniably clear.

These aren't rules to memorize. They're questions to ask yourself every time you make a chart.

### Principle 1: Lead with the Insight, Not the Chart

**Bad:** "Here is the salary distribution across departments."

**Good:** "Engineering earns 34% more than HR — and the gap widens with experience."

The chart proves your point. The point comes first.
In ML: don't say "here is the confusion matrix" — say "the model misses 40% of churners, which is the expensive error."

### Principle 2: One Chart = One Message

If you need to say two things, make two charts.
If your chart needs a paragraph to explain, it's doing too much.

The test: can you write the chart's title as a single declarative sentence?
- ✗ "Salary, Experience, Age, and Department Analysis"
- ✓ "Senior engineers in Engineering earn 2x more than entry-level"

### Principle 3: Remove Everything That Doesn't Earn Its Place

In [14]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

np.random.seed(42)
df = pd.DataFrame({
    'department': ['Engineering', 'Sales', 'Marketing', 'HR'],
    'avg_salary': [95000, 72000, 68000, 61000]
})

# CLUTTERED VERSION — gridlines, 3D effect, redundant legend
fig_bad = go.Figure(go.Bar(
    x=df['department'], y=df['avg_salary'],
    marker_color=['blue', 'orange', 'green', 'red'],  # random colors add nothing
))
fig_bad.update_layout(title='BAD: Cluttered — colors encode no meaning, no clear point')
fig_bad.show()

# CLEAN VERSION — highlight the one thing that matters
highlight_color = ['#1f77b4' if d == 'Engineering' else '#d3d3d3' for d in df['department']]

fig_good = go.Figure(go.Bar(
    x=df['department'],
    y=df['avg_salary'],
    marker_color=highlight_color,
    text=df['avg_salary'].apply(lambda x: f'${x:,.0f}'),
    textposition='outside'
))
fig_good.update_layout(
    title='GOOD: Engineering pays 56% more than HR',
    yaxis=dict(showgrid=False, showticklabels=False, title=''),  # y-axis redundant — values on bars
    xaxis_title='',
    template='plotly_white',
    showlegend=False
)
fig_good.show()

**What was removed and why:**
- Random colors → gray + one accent color (color now means something)
- Y-axis ticks → removed (values are on the bars, ticks are redundant)
- Vague title → replaced with the actual insight

The rule: if removing something doesn't lose information, remove it.

### Principle 4: Color is Your Most Powerful Tool — Use It Sparingly

In [15]:
# Color should answer: "what do you want the reader to look at first?"

# Pattern: gray everything, accent the thing that matters
departments = ['Engineering', 'Sales', 'Marketing', 'HR']
salaries = [95000, 72000, 68000, 61000]
churn_rate = [0.12, 0.28, 0.22, 0.31]  # HR has highest churn

# Highlight the department with a problem
colors = ['#d3d3d3' if d != 'HR' else '#e74c3c' for d in departments]

fig = go.Figure(go.Bar(
    x=departments, y=churn_rate,
    marker_color=colors,
    text=[f'{r:.0%}' for r in churn_rate],
    textposition='outside'
))
fig.update_layout(
    title='HR has the highest churn — nearly 1 in 3 employees leave',
    yaxis=dict(showticklabels=False, showgrid=False),
    template='plotly_white',
    showlegend=False
)
fig.show()

**Color rules for ML/data work:**
- 1 accent color + gray for everything else
- Red = bad/alert, Green = good, Blue = neutral (use these conventions, don't fight them)
- Don't use color to decorate — use it to direct attention
- For sequential data (low → high): single-color gradient
- For diverging data (-1 to +1, like correlation): blue-white-red (`RdBu`)

### Principle 5: Annotate on the Chart, Not in a Legend

In [16]:
# Legends make readers look back and forth. Direct labels keep eyes on the data.

epochs = list(range(1, 21))
np.random.seed(0)
train_loss = [1.0 * (0.85 ** i) + np.random.normal(0, 0.015) for i in epochs]
val_loss   = [1.0 * (0.87 ** i) + np.random.normal(0, 0.025) for i in epochs]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=epochs, y=train_loss,
    mode='lines', name='Train',
    line=dict(color='#1f77b4', width=2)
))
fig.add_trace(go.Scatter(
    x=epochs, y=val_loss,
    mode='lines', name='Validation',
    line=dict(color='#ff7f0e', width=2, dash='dash')
))

# Annotate directly on the lines instead of relying on legend
fig.add_annotation(x=19, y=train_loss[-1], text='Train', showarrow=False,
                   xanchor='left', font=dict(color='#1f77b4'))
fig.add_annotation(x=19, y=val_loss[-1], text='Val', showarrow=False,
                   xanchor='left', font=dict(color='#ff7f0e'))

# Highlight where overfitting starts
fig.add_vline(x=12, line_dash='dot', line_color='red', opacity=0.5)
fig.add_annotation(x=12, y=max(val_loss)*0.9, text='Overfitting starts',
                   showarrow=True, arrowhead=1, font=dict(color='red'))

fig.update_layout(
    title='Model begins overfitting after epoch 12',
    xaxis_title='Epoch', yaxis_title='Loss',
    template='plotly_white',
    showlegend=False  # direct labels replace legend
)
fig.show()

### Principle 6: Sort Your Bars (Almost Always)

In [17]:
feature_importance = pd.DataFrame({
    'feature': ['age', 'experience', 'department', 'salary_lag', 'tenure', 'performance_score'],
    'importance': [0.31, 0.28, 0.17, 0.12, 0.08, 0.04]
})

# Alphabetical order (BAD — readers have to search for the most important feature)
fig_bad = px.bar(feature_importance, x='feature', y='importance',
                 title='BAD: Alphabetical order — hard to read')
fig_bad.show()

# Sorted by importance (GOOD)
fig_good = px.bar(
    feature_importance.sort_values('importance'),
    x='importance', y='feature',
    orientation='h',           # horizontal works better for long feature names
    title='GOOD: Age and experience drive the model — sorted by impact'
)
fig_good.update_layout(template='plotly_white', yaxis_title='')
fig_good.show()

### Principle 7: Choose the Right Chart Type

| You want to show | Use |
|---|---|
| Distribution of one variable | Histogram, Box plot, Violin |
| Relationship between two variables | Scatter plot |
| Comparison across categories | Bar chart (sorted) |
| Change over time / epochs | Line chart |
| Part of a whole | Stacked bar (not pie — hard to compare) |
| Correlation matrix | Heatmap |
| Feature importance | Horizontal bar (sorted) |
| Confusion matrix | Heatmap with annotations |
| Threshold tradeoff (precision/recall) | Line chart with two y-axes |

**Never use:** 3D charts, pie charts with >3 slices, dual y-axis (almost always misleads)

### Principle 8: The Question to Ask Before Every Chart

**"So what?"**

If you show this chart to a stakeholder or in a PR review and they say "so what?" — your chart is describing data, not communicating insight.

Before you finalize any chart, complete this sentence:
> "This chart shows that _______, which means we should _______."

If you can't complete it, rethink what you're visualizing.

**ML-specific examples:**
- ✓ "The model's precision drops below 0.8 at threshold > 0.6 — we should keep threshold ≤ 0.55"
- ✓ "Experience explains 28% of churn variance — it should be our top feature"
- ✗ "Here is the ROC curve" (what should someone do with this?)

### Putting It Together: ML Model Report Chart

In [19]:
# Example: A complete model evaluation chart that tells a story
from plotly.subplots import make_subplots

thresholds = np.arange(0.1, 1.0, 0.05)
precision  = 0.5 + 0.5 * thresholds + np.random.normal(0, 0.02, len(thresholds))
recall     = 1.0 - 0.8 * thresholds + np.random.normal(0, 0.02, len(thresholds))
precision  = np.clip(precision, 0, 1)
recall     = np.clip(recall, 0, 1)

fig = go.Figure()

fig.add_trace(go.Scatter(x=thresholds, y=precision, name='Precision',
                         line=dict(color='#1f77b4', width=2)))
fig.add_trace(go.Scatter(x=thresholds, y=recall, name='Recall',
                         line=dict(color='#ff7f0e', width=2, dash='dash')))

# Highlight the chosen operating point
chosen_threshold = 0.40
fig.add_vline(x=chosen_threshold, line_dash='dot', line_color='green')
fig.add_annotation(
    x=chosen_threshold, y=0.5,
    text=f'Operating point<br>threshold={chosen_threshold}',
    showarrow=True, arrowhead=2, arrowcolor='green',
    font=dict(color='green')
)

fig.update_layout(
    title='Threshold 0.40 balances catching churners without overwhelming retention team',
    xaxis_title='Decision Threshold',
    yaxis_title='Score',
    template='plotly_white',
    yaxis=dict(range=[0, 1.05])
)
fig.show()

---
## Summary

| Topic | Key Takeaway |
|---|---|
| **Plotly** | Use `px` for 90% of charts. Use `go` when you need full control. |
| **Streamlit** | Re-runs top to bottom on every interaction. Cache expensive ops. Build to learn. |
| **Storytelling** | Lead with insight → one message per chart → remove clutter → color directs attention → annotate directly → sort bars → ask "so what?" |

The visualization skill that matters most for an AI/ML engineer isn't aesthetics — it's the ability to make a model's behavior legible to someone who didn't build it.